In [2]:
#!/usr/bin/env python3
import argparse
import glob
import os
from typing import List, Dict, Any

import pandas as pd

# Optional but faster for Parquet
try:
    import pyarrow as pa  # noqa: F401
    import pyarrow.dataset as ds
    HAVE_ARROW = True
except Exception:
    HAVE_ARROW = False


def load_parquet_any(path: str) -> pd.DataFrame:
    """
    Load a single parquet file or a directory or a glob of many parquet files.
    Prefers pyarrow.dataset for speed if available.
    """
    if os.path.isdir(path):
        if HAVE_ARROW:
            dataset = ds.dataset(path, format="parquet")
            tbl = dataset.to_table()
            return tbl.to_pandas(split_blocks=True, self_destruct=True)
        else:
            files = glob.glob(os.path.join(path, "**/*.parquet"), recursive=True)
            return _concat_parquet(files)
    else:
        # treat as glob or single file
        files = sorted(glob.glob(path))
        if len(files) == 0:
            raise FileNotFoundError(f"No parquet files match: {path}")
        if HAVE_ARROW and len(files) > 1:
            dataset = ds.dataset(files, format="parquet")
            tbl = dataset.to_table()
            return tbl.to_pandas(split_blocks=True, self_destruct=True)
        return _concat_parquet(files)


def _concat_parquet(files: List[str]) -> pd.DataFrame:
    dfs = [pd.read_parquet(f) for f in files]
    return pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]


def ensure_list(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return x
    # some loaders store sequences as strings like "['CWE-78']"
    if isinstance(x, str):
        # try to parse by splitting, fall back to eval if it looks like a list
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                import ast
                val = ast.literal_eval(s)
                return val if isinstance(val, list) else [str(val)]
            except Exception:
                return [s]
        # comma or semicolon separated
        if "," in s:
            return [t.strip() for t in s.split(",") if t.strip()]
        if ";" in s:
            return [t.strip() for t in s.split(";") if t.strip()]
        return [s]
    return [x]


def normalize_context_row(ctx: Any) -> Dict[str, List[str]]:
    """
    context is a struct with keys like:
      - Execution Environment (sequence of string)
      - Explanation (sequence of string)
      - External Function (sequence of string)
      - Function Argument (sequence of string)
      - Globals (sequence of string)
      - Type Execution Declaration (sequence of string)
    Normalize to dict of lists. Empty lists for missing.
    """
    keys = [
        "Execution Environment",
        "Explanation",
        "External Function",
        "Function Argument",
        "Globals",
        "Type Execution Declaration",
    ]
    out = {}
    if isinstance(ctx, dict):
        for k in keys:
            out[k] = ensure_list(ctx.get(k, []))
    else:
        for k in keys:
            out[k] = []
    return out


def basic_eda(df: pd.DataFrame) -> None:
    print("\n=== Shape and dtypes ===")
    print(df.shape)
    print(df.dtypes)

    print("\n=== Null counts (top 20) ===")
    print(df.isna().sum().sort_values(ascending=False).head(20))

    print("\n=== Head ===")
    print(df.head(3))

    # Basic distributions
    print("\n=== Vulnerable ratio ===")
    if "is_vulnerable" in df.columns:
        print(df["is_vulnerable"].value_counts(dropna=False, normalize=True))

    print("\n=== Top projects ===")
    if "project" in df.columns:
        print(df["project"].value_counts().head(10))

    print("\n=== Top repos ===")
    if "project_url" in df.columns:
        print(df["project_url"].value_counts().head(10))

    # Text length heuristics
    if "func_body" in df.columns:
        df["_func_len"] = df["func_body"].fillna("").map(len)
        print("\nfunc_body length stats:")
        print(df["_func_len"].describe())

    # CWE counts
    if "cwe_list" in df.columns:
        s = df["cwe_list"].explode().dropna().astype(str)
        print("\nTop CWEs:")
        print(s.value_counts().head(15))

    # CVE counts
    if "cve_list" in df.columns:
        s = df["cve_list"].explode().dropna().astype(str)
        print("\nTop CVEs:")
        print(s.value_counts().head(15))

    # Vulnerability by project (top)
    if {"project", "is_vulnerable"}.issubset(df.columns):
        agg = (
            df.groupby("project")["is_vulnerable"]
            .mean()
            .sort_values(ascending=False)
            .head(15)
        )
        print("\nProjects with highest vulnerable rate (min 20 samples):")
        counts = df.groupby("project").size()
        mask = counts[counts >= 20].index
        print(agg[agg.index.isin(mask)])


def save_quick_artifacts(df: pd.DataFrame, out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)

    # Top CWEs
    if "cwe_list" in df.columns:
        cwe_counts = df["cwe_list"].explode().dropna().astype(str).value_counts()
        cwe_counts.to_csv(os.path.join(out_dir, "cwe_counts.csv"))

    # Vulnerable by project
    if {"project", "is_vulnerable"}.issubset(df.columns):
        proj = (
            df.groupby("project")["is_vulnerable"]
            .agg(["mean", "count"])
            .sort_values("count", ascending=False)
        )
        proj.to_csv(os.path.join(out_dir, "project_vuln_stats.csv"))

    # Sample rows for quick inspection
    cols = [
        "idx", "project", "project_url", "filepath", "commit_id",
        "is_vulnerable", "cve_list", "cwe_list", "func_name", "func_body"
    ]
    sample_cols = [c for c in cols if c in df.columns]
    df.sample(n=min(25, len(df)), random_state=0)[sample_cols] \
      .to_csv(os.path.join(out_dir, "sample_rows.csv"), index=False)
    print(f"\nSaved artifacts to: {out_dir}")


In [3]:
import os 
print(os.getcwd())
df = load_parquet_any("data/train-00000-of-00001.parquet")
basic_eda(df)

/home/ema8/UTSV-8580/secvul-llm-study/datasets/SecVulEval

=== Shape and dtypes ===
(25440, 16)
idx                     int64
project                object
project_url            object
filepath               object
commit_id              object
commit_message         object
is_vulnerable            bool
hash                   object
func_name              object
func_body              object
changed_lines          object
changed_statements     object
cve_list               object
cwe_list               object
fixed_func_idx        float64
context                object
dtype: object

=== Null counts (top 20) ===
fixed_func_idx        5392
commit_message         102
project_url              0
idx                      0
filepath                 0
commit_id                0
is_vulnerable            0
project                  0
hash                     0
func_name                0
changed_lines            0
func_body                0
changed_statements       0
cve_list                 0
cw

In [4]:
if 'context' in df.columns:
    print("Normalizing context for all rows... (this may take a moment)")
    df['context_norm'] = df['context'].apply(normalize_context_row)
    print("Done. Sample:")

else:
    print("No 'context' column found in df")

for idx, ctx in df['context_norm'].head().items():
    print(f"Index: {idx}")
    for key, value in ctx.items():
        print(f"  {key}: {value}")



Normalizing context for all rows... (this may take a moment)
Done. Sample:
Index: 0
  Execution Environment: [array(['CONFIG_VMAP_STACK'], dtype=object)]
  Explanation: []
  External Function: [array(['usb_get_intfdata', 'dev_info'], dtype=object)]
  Function Argument: [array(['intf'], dtype=object)]
  Globals: [array(['KBUILD_MODNAME'], dtype=object)]
  Type Execution Declaration: [array(['struct dvb_usb_device', 'struct device'], dtype=object)]
Index: 1
  Execution Environment: [array(['CONFIG_VMAP_STACK'], dtype=object)]
  Explanation: []
  External Function: [array(['usb_get_intfdata', 'dev_info'], dtype=object)]
  Function Argument: [array(['intf'], dtype=object)]
  Globals: [array(['KBUILD_MODNAME'], dtype=object)]
  Type Execution Declaration: [array(['struct dvb_usb_device', 'struct device'], dtype=object)]
Index: 2
  Execution Environment: [array([], dtype=object)]
  Explanation: []
  External Function: [array(['nfp_cpp_area_release', 'cpp->op->area_init',
       'nfp_cpp_area

In [5]:
# Count how many rows have the same "fixed_func_idx"
df['fixed_func_idx'].value_counts()  # Seems to all be 2 

fixed_func_idx
25438.0    2
1.0        2
3.0        2
6.0        2
8.0        2
          ..
24.0       2
26.0       2
28.0       2
30.0       2
32.0       2
Name: count, Length: 10024, dtype: int64

In [6]:
# For each fixed_func_idx ensure 1 has `is_vulnerable` True and 1 False
grouped = df.groupby('fixed_func_idx')['is_vulnerable'].agg(['sum', 'count'])
print(grouped[grouped['sum'] != 1])  # Should be empty

grouped = df.groupby('fixed_func_idx')
print(grouped.head())

Empty DataFrame
Columns: [sum, count]
Index: []
         idx  project                                        project_url  \
0          0    linux                  https://github.com/torvalds/linux   
1          1    linux                  https://github.com/torvalds/linux   
2          2    linux                  https://github.com/torvalds/linux   
3          3    linux                  https://github.com/torvalds/linux   
4          4    linux                  https://github.com/torvalds/linux   
...      ...      ...                                                ...   
25432  25432  Android  https://android.googlesource.com/platform/exte...   
25435  25435  Android  https://android.googlesource.com/platform/hard...   
25436  25436  Android  https://android.googlesource.com/platform/hard...   
25437  25437  Android  https://android.googlesource.com/platform/syst...   
25438  25438  Android  https://android.googlesource.com/platform/syst...   

                                       

In [8]:
import numpy as np
import pandas as pd

def pair_vulnerable_and_fixed(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforms the row-per-version DataFrame (one row for vuln, one for fixed) 
    into a single row per bug fix (vuln code and fixed code side-by-side).
    
    This replicates the desired structure from your previous Clang loop.
    """
    print("\n--- Starting Pairing and Pivoting ---")
    
    # Check for the required column
    if 'fixed_func_idx' not in df.columns:
        raise ValueError("DataFrame must contain 'fixed_func_idx' column for grouping.")

    # Select columns needed for the final output, plus the grouping and flag columns
    pivot_cols = [
        "fixed_func_idx", "is_vulnerable", "func_body", "func_name", 
        "cve_list", "cwe_list", "context_norm" 
    ]
    
    # Ensure only columns present in the dataframe are used
    available_cols = [c for c in pivot_cols if c in df.columns]
    df_pairs = df.dropna(subset=['func_body', 'fixed_func_idx'])[available_cols].copy()

    # CRITICAL STEP 1: Select one representative row for each version/index pair
    # This prevents the Cartesian product explosion seen earlier.
    df_vuln = df_pairs[df_pairs['is_vulnerable'] == True].groupby('fixed_func_idx').first().reset_index()
    df_fixed = df_pairs[df_pairs['is_vulnerable'] == False].groupby('fixed_func_idx').first().reset_index()
    
    # CRITICAL STEP 2: Merge them side-by-side on the unique ID
    df_merged = df_vuln.merge(
        df_fixed, 
        on='fixed_func_idx', 
        how='inner',  # Only keep perfect pairs
        suffixes=('_vuln', '_fixed')
    )

    # CRITICAL STEP 3: Select and rename columns to match the LLM input format
    df_final = pd.DataFrame({
        "func_name": df_merged['func_name_vuln'],
        "fixed_func_idx": df_merged['fixed_func_idx'],
        "cve_list": df_merged['cve_list_vuln'],
        "cwe_list": df_merged['cwe_list_vuln'],
        "vuln_func_body": df_merged['func_body_vuln'],
        "fixed_func_body": df_merged['func_body_fixed'],
        "context_norm": df_merged['context_norm_vuln'], # Keep the vulnerable context for base analysis
    })
    
    print(f"Final merged pairs ready for LLM: {len(df_final)}")
    
    return df_final.reset_index(drop=True)

# --- EXAMPLE USAGE (Insert this after your initial data loading cell) ---
# Assuming 'df' is the large DataFrame loaded in your notebook's first cell.

# df_paired = pair_vulnerable_and_fixed(df)

# Now, your LLM filter script (run_llm_filter) will iterate over df_paired 
# instead of trying to access 'grouped'.

#pass grouped into that and save the output
df_paired = pair_vulnerable_and_fixed(df)
#save this as a csv
df_paired.to_csv("data/paired_vuln_fixed.csv", index=False)


--- Starting Pairing and Pivoting ---
Final merged pairs ready for LLM: 10024


In [40]:
from clang.cindex import Index, CursorKind, TypeKind
from tqdm import tqdm
import pandas as pd
import os

# ==========================================
# 1. CONFIGURATION & LISTS
# ==========================================

# Toggle: "STRICT" (rejects malloc/logging) vs "LOOSE" (allows them via harness)
FILTER_MODE = "LOOSE" 
MAX_UNRESOLVED_DIAGNOSTICS = 3 # Hard limit on Clang "undeclared identifier" errors

print(f"--- RUNNING IN {FILTER_MODE} MODE ---")

# A. CORE DENY LIST (Semantic AST Check)
# Always rejected via AST analysis (locks, hardware, ASM, VFS)
CORE_DENY_PREFIXES = [
    "copy_from_user", "copy_to_user", "get_user", "put_user", "access_ok",
    "schedule", "msleep", "mutex_", "spin_", "rwlock_", "rcu_", "atomic_", "completion",
    "pci_", "usb_", "vfio_", "mlx5_", "nfp_", "ioread", "iowrite", "dma_", "outb", "inb",
    "request_irq", "free_irq", "tasklet_", "clk_",
    "panic", "BUG", "oops", "unreachable",
    "vfs_", "filp_", "inode_", "dentry_", "path_", "sock_", "sk_", "netdev_"
]
CORE_DENY_EXACT = {"capable", "ioctl", "syscall", "asm", "__asm__"}

# B. OPTIONAL DENY LIST (Memory/Logging)
OPTIONAL_DENY_PREFIXES = [
    "kmalloc", "kzalloc", "kcalloc", "kvzalloc", "vmalloc", "memdup", 
    "kfree", "kvfree", "printk", "pr_", "dev_" 
]

# C. TEXT DENY LIST (Fast Heuristic Check)
# These defeat the AST (macros/obfuscation), so we search raw text first.
# C. TEXT DENY LIST (Fast Heuristic Check)
# C. TEXT DENY LIST (Fast Heuristic Check)
TEXT_DENY_KEYWORDS = (
    "gss_", "OM_", "GSS_",          # Security
    "mpi_", "MPI", "MPI_",          # Parallel/Crypto
    "AVFilter", "AVFrame", "ff_",   # FFmpeg
    "Thunar",                       # File Manager
    "rdp", "freerdp",               # RDP (Remote Desktop)
    "rfb", "vnc"                    # <--- NEW: VNC (Remote Frame Buffer)
    # Add C++ and Framework specific keywords
    "TTypes", "Device&", "operator()", "template", "typename"
)

# Build Active AST Lists
if FILTER_MODE == "STRICT":
    ACTIVE_DENY_PREFIXES = tuple(CORE_DENY_PREFIXES + OPTIONAL_DENY_PREFIXES)
else:
    ACTIVE_DENY_PREFIXES = tuple(CORE_DENY_PREFIXES)

ACTIVE_DENY_EXACT = CORE_DENY_EXACT

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def get_context_list(ctx_data, key):
    """Safely extracts list data from the context_norm NumPy/Pandas structure."""
    try:
        if key in ctx_data and ctx_data[key] is not None:
            # Check if it's a list/array with at least one item
            val = ctx_data[key]
            if len(val) > 0:
                # It might be [array(['item'], dtype=object)]
                first_item = val[0]
                if hasattr(first_item, 'tolist'):
                    return first_item.tolist()
                if isinstance(first_item, (list, tuple)):
                    return list(first_item)
    except Exception:
        pass
    return []

def is_disallowed_identifier(name: str) -> bool:
    if not name: return False
    if name in ACTIVE_DENY_EXACT: return True
    if name.startswith(ACTIVE_DENY_PREFIXES): return True
    return False

# ==========================================
# 3. CLANG CLASSIFICATION ENGINE
# ==========================================

def classify_function(func_src: str, idx: Index, max_diag: int):
    tu = idx.parse('tmp.c', args=['-std=c11', '-D__KERNEL__'],
                   unsaved_files=[('tmp.c', func_src)], options=0)

    # 1. DIAGNOSTIC THRESHOLD CHECK (Hard Deny Failsafe)
    unresolved_count = 0
    for diag in tu.diagnostics:
        if diag.severity >= 3 and ("undeclared" in diag.spelling.lower() or "implicit" in diag.spelling.lower()):
            unresolved_count += 1

    if unresolved_count > max_diag:
        return {
            'category': 'hard',
            'reasons': [f"DIAGNOSTIC_FAIL: {unresolved_count} unresolved symbols (Limit {max_diag})"],
            'prototype': 'N/A', 'globals_used': [], 'structs_used': []
        }

    # 2. FIND FUNCTION DECLARATION
    fn = next((c for c in tu.cursor.get_children() if c.kind == CursorKind.FUNCTION_DECL), None)
    
    if fn is None: 
        return {
            'category': 'hard', 'reasons': ["Parse Error (AST Failure)"], 
            'prototype': 'N/A', 'globals_used': [], 'structs_used': []
        }

    reasons = []
    globals_used = set()
    structs_used = set()
    is_hard = False
    
    # 3. PROTOTYPE ANALYSIS
    param_strs = []
    for ch in fn.get_children():
        if ch.kind == CursorKind.PARM_DECL:
            param_strs.append(f"{ch.type.spelling} {ch.spelling}")
            
            # Check for structs/complex types
            if ch.type.kind in (CursorKind.STRUCT_DECL, TypeKind.RECORD, TypeKind.TYPEDEF, TypeKind.POINTER):
                is_unresolved_pointer = False
                if ch.type.kind == TypeKind.POINTER:
                    if ch.type.get_pointee().kind in (TypeKind.UNEXPOSED, TypeKind.INVALID):
                        is_unresolved_pointer = True

                tname = ch.type.spelling.replace('*', '').strip()
                
                # Filter out standard C types
                if is_unresolved_pointer or tname not in (
                    "int", "char", "void", "long", "size_t", "short", "float", "double", "bool",
                    "u8", "u16", "u32", "u64", "s8", "s16", "s32", "s64",
                    "unsigned int", "unsigned long", "unsigned char", "const"):
                    
                    if is_unresolved_pointer:
                         structs_used.add(f"UNRESOLVED_TYPE_{ch.spelling}")
                    else:
                         structs_used.add(tname)

    proto = f"{fn.result_type.spelling} {fn.spelling}({', '.join(param_strs)})"

    # 4. BODY ANALYSIS
    def walk(n):
        nonlocal is_hard
        if is_hard: return 
        
        # Check Calls / Vars / Types against Deny List
        name = None
        if n.kind == CursorKind.CALL_EXPR:
            callee = next((c for c in n.get_children() if c.kind in (CursorKind.DECL_REF_EXPR, CursorKind.MEMBER_REF_EXPR)), None)
            if callee: name = callee.spelling
        elif n.kind == CursorKind.DECL_REF_EXPR and n.referenced:
            name = n.referenced.spelling
        elif n.kind == CursorKind.TYPE_REF:
            name = n.spelling

        if name and is_disallowed_identifier(name):
            is_hard = True; reasons.append(f"Deny Match: {name}"); return

        # Check ASM / IOMEM
        if n.kind == CursorKind.ASM_STMT:
            is_hard = True; reasons.append("Inline Assembly"); return
        if n.kind == CursorKind.TYPE_REF and n.type and ("__iomem" in n.type.spelling):
            is_hard = True; reasons.append("Usage of __iomem"); return

        # Globals Detection
        if n.kind == CursorKind.DECL_REF_EXPR and n.referenced:
            if n.referenced.kind == CursorKind.VAR_DECL:
                # Variable defined outside function extent
                if not (fn.extent.start.file == n.referenced.extent.start.file and
                        fn.extent.start.offset <= n.referenced.extent.start.offset <= fn.extent.end.offset):
                    globals_used.add(n.referenced.spelling)

        for ch in n.get_children():
            walk(ch)
            
    walk(fn)

    # 5. BUCKET LOGIC
    if is_hard:
        return {'category': 'hard', 'prototype': proto, 'reasons': reasons, 'globals_used': list(globals_used), 'structs_used': list(structs_used)}

    if len(globals_used) > 0 or len(structs_used) > 0:
        medium_reasons = []
        if globals_used: medium_reasons.append("Uses Globals")
        if structs_used: medium_reasons.append("Complex Params/Structs")
        return {'category': 'medium', 'prototype': proto, 'reasons': medium_reasons, 'globals_used': list(globals_used), 'structs_used': list(structs_used)}

    return {'category': 'easy', 'prototype': proto, 'reasons': ["Pure Logic"], 'globals_used': [], 'structs_used': []}

# ==========================================
# 4. MAIN PROCESSING LOOP
# ==========================================

idx = Index.create()
buckets = {"easy": [], "medium": [], "hard": []}
total_pairs = 0

print("Starting hybrid classification...")

for name, group in tqdm(grouped):
    total_pairs += 1
    
    vuln_func_rows = group[group['is_vulnerable'] == True]
    fixed_func_rows = group[group['is_vulnerable'] == False]
    
    if len(vuln_func_rows) != 1 or len(fixed_func_rows) != 1: continue
        
    vuln_func = vuln_func_rows.iloc[0]
    vuln_body = vuln_func['func_body']
    fixed_func = fixed_func_rows.iloc[0]
    
    # --- A. FAST TEXT SEARCH ---
    is_hard_deny_text = False
    for keyword in TEXT_DENY_KEYWORDS:
        if keyword in vuln_body:
            is_hard_deny_text = True
            result = {'category': 'hard', 'reasons': [f"TEXT_DENY: Found '{keyword}'"], 
                      'prototype': vuln_func['func_name'], 'globals_used': [], 'structs_used': []}
            break
            
    # --- B. CLANG ANALYSIS ---
    if not is_hard_deny_text:
        result = classify_function(vuln_body, idx, MAX_UNRESOLVED_DIAGNOSTICS)
        cat = result['category']
    else:
        cat = 'hard'

    # --- C. CONTEXT AUGMENTATION ---
    try:
        ctx_data = df.loc[name, 'context_norm']
        
        context_globals = get_context_list(ctx_data, 'Globals')
        result['globals_used'].extend(context_globals)
        result['globals_used'] = list(set(result['globals_used']))
        
        context_structs = get_context_list(ctx_data, 'Type Execution Declaration')
        result['structs_used'].extend(context_structs)
        result['structs_used'] = list(set(result['structs_used']))
    except KeyError:
        pass 

    # --- D. FINAL CATEGORY CHECK ---
    if cat != 'hard':
        if len(result['globals_used']) > 0 or len(result['structs_used']) > 0:
            if cat == 'easy': result['reasons'].append("UPGRADED: Context Data found complexity.")
            cat = 'medium'

    # --- E. SAVE ---
    buckets[cat].append({
        "func_name": vuln_func['func_name'],
        "fixed_func_idx": name,
        "function_prototype": result.get('prototype', 'N/A'),
        "category": cat,
        "classification_reasons": "; ".join(result['reasons']),
        "globals_detected": "; ".join(result['globals_used']),
        "structs_detected": "; ".join(result['structs_used']),
        "cve_list": vuln_func['cve_list'],
        "cwe_list": vuln_func['cwe_list'],
        "vuln_func_body": vuln_body,
        "fixed_func_body": fixed_func['func_body'],
    })

--- RUNNING IN LOOSE MODE ---
Starting hybrid classification...


  0%|          | 0/10024 [00:00<?, ?it/s]

100%|██████████| 10024/10024 [00:35<00:00, 281.10it/s]


In [41]:
# ==========================================
# 5. WRITE OUTPUT TO FILES
# ==========================================

# Define the output directory based on your filter mode
output_dir = f"data_triaged_{FILTER_MODE.lower()}"
os.makedirs(output_dir, exist_ok=True)

print(f"\n--- Saving results to '{output_dir}/' ---")

total_saved = 0
for cat, rows in buckets.items():
    # Convert the list of dicts to a DataFrame
    out_df = pd.DataFrame(rows)
    count = len(out_df)
    total_saved += count
    
    print(f"  Bucket [{cat.upper()}]: {count} samples")
    
    if count > 0:
        # Save to CSV
        filename = os.path.join(output_dir, f"dataset_{cat}.csv")
        out_df.to_csv(filename, index=False)
        print(f"    -> Saved {filename}")

print(f"\nTotal samples processed and saved: {total_saved}")

# --- Debug Check ---
if total_saved == 0:
    print("\n[WARNING] No samples were saved. This might mean:")
    print("1. The 'grouped' iterator was empty.")
    print("2. The 'buckets' dictionary was reset.")
    print("3. An error occurred inside the loop preventing appending.")


--- Saving results to 'data_triaged_loose/' ---
  Bucket [EASY]: 45 samples
    -> Saved data_triaged_loose/dataset_easy.csv
  Bucket [MEDIUM]: 520 samples
    -> Saved data_triaged_loose/dataset_medium.csv
  Bucket [HARD]: 9459 samples
    -> Saved data_triaged_loose/dataset_hard.csv

Total samples processed and saved: 10024


Results for STRICT mode:
  [EASY]: 1720 samples
  [MEDIUM]: 4805 samples
  [HARD]: 3499 samples
Saved to data_triaged_strict

In [8]:
from clang.cindex import Index, CursorKind, TypeKind
from tqdm import tqdm
import pandas as pd
import os

# --- (Your DENY lists remain the same) ---
DENY_PREFIXES = (
    "copy_from_user", "copy_to_user", "get_user", "put_user",
    "printk", "pr_", "dev_", "netlink_", "schedule", "msleep",
    "mutex_", "spin_lock", "rwlock_", "rcu_", "atomic_",
    "pci_", "usb_", "vfio_", "mlx5_", "nfp_", "ioread", "iowrite",
    "request_irq", "free_irq",
    "kmalloc", "kzalloc", "kcalloc", "kvzalloc", "vmalloc", "memdup_user",
    "kfree", "kvfree",
)
DENY_EXACT = {"capable", "ioctl"}
DENY_TYPES = {"__iomem"}

# --- MODIFIED FUNCTION ---
def check_unit_testable_and_get_proto(func_src: str) -> tuple[bool, str | None]:
    """
    Parses the function ONCE.
    Checks all rules, and if they pass, returns (True, "function_prototype_string").
    If any rule fails, it returns (False, None).
    """
    idx = Index.create()
    tu = idx.parse('tmp.c',
                   args=['-std=c11', '-D__KERNEL__'],
                   unsaved_files=[('tmp.c', func_src)],
                   options=0)

    # Find the first function in this snippet
    fn = None
    for c in tu.cursor.get_children():
        if c.kind == CursorKind.FUNCTION_DECL:
            fn = c
            break
    if fn is None:
        return (False, None)  # could not parse a function

    # --- Rule 1: Reject void return ---
    if fn.result_type.kind == TypeKind.VOID:
        return (False, None)

    # --- Rule 2: Must have at least one parameter ---
    has_param = any(ch.kind == CursorKind.PARM_DECL for ch in fn.get_children())
    if not has_param:
        return (False, None)
        
    # --- Build prototype (since we have the 'fn' object already) ---
    param_types = []
    for ch in fn.get_children():
        if ch.kind == CursorKind.PARM_DECL:
            param_types.append(ch.type.spelling)
    proto = f"{fn.result_type.spelling} {fn.spelling}({', '.join(param_types)})"
    # --- End prototype logic ---

    has_return_expr = False
    disallowed_hit = False

    def is_disallowed_call(name: str) -> bool:
        return name in DENY_EXACT or name.startswith(DENY_PREFIXES)

    def walk(n, scope_stack):
        nonlocal has_return_expr, disallowed_hit
        # Detect return with expression
        if n.kind == CursorKind.RETURN_STMT:
            if any(True for _ in n.get_children()):
                has_return_expr = True

        # Detect function calls
        if n.kind == CursorKind.CALL_EXPR:
            callee = next((c for c in n.get_children()
                           if c.kind in (CursorKind.DECL_REF_EXPR, CursorKind.MEMBER_REF_EXPR)), None)
            if callee and callee.spelling and is_disallowed_call(callee.spelling):
                disallowed_hit = True

        # Detect global variable references
        if n.kind == CursorKind.DECL_REF_EXPR and n.referenced:
            if n.referenced.kind == CursorKind.VAR_DECL:
                if not (fn.extent.start.file == n.referenced.extent.start.file and
                        fn.extent.start.offset <= n.referenced.extent.start.offset <= fn.extent.end.offset):
                    disallowed_hit = True

        # Crude qualifier checks for MMIO/volatile
        if n.kind == CursorKind.TYPE_REF and n.type and ("__iomem" in n.type.spelling):
            disallowed_hit = True
        if n.kind == CursorKind.UNEXPOSED_EXPR and "volatile" in n.displayname:
            disallowed_hit = True

        for ch in n.get_children():
            walk(ch, scope_stack)

    walk(fn, [])

    # --- Rule 3: Require at least one `return expr;` ---
    if not has_return_expr:
        return (False, None)
        
    # --- Rule 4: Check deny lists ---
    if disallowed_hit:
        return (False, None)
        
    # --- All rules passed ---
    return (True, proto)

In [9]:
from tqdm import tqdm
import pandas as pd
import os

# Define the columns for the output DataFrame
columns = ["func_name", "fixed_func_idx", "function_prototype", "cve_list", "cwe_list", "vuln_func_body", "fixed_func_body"]

# --- RE-INTRODUCING THE 'NOT TESTABLE' LIST ---
# This is critical for debugging our filter.
unit_testable_builder = []
not_unit_testable_builder = []

total_pairs_processed = 0

for name, group in tqdm(grouped):
    total_pairs_processed += 1
    
    vuln_func_rows = group[group['is_vulnerable'] == True]
    fixed_func_rows = group[group['is_vulnerable'] == False]

    # Robustness check: Ensure we have exactly one of each
    if len(vuln_func_rows) != 1 or len(fixed_func_rows) != 1:
        # print(f"Skipping group {name}: not a perfect pair.")
        continue
        
    vuln_func = vuln_func_rows.iloc[0]
    fixed_func = fixed_func_rows.iloc[0]
    
    vuln_body = vuln_func['func_body']

    # --- EFFICIENT CHECK ---
    # This one call does everything and only parses the code ONCE
    is_testable, prototype = check_unit_testable_and_get_proto(vuln_body)

    # --- MODIFIED LOGIC TO SAVE BOTH ---
    if is_testable:
        unit_testable_builder.append({
            "func_name": vuln_func['func_name'],
            "fixed_func_idx": name,
            "function_prototype": prototype, # Use the prototype we just got
            "cve_list": vuln_func['cve_list'],
            "cwe_list": vuln_func['cwe_list'],
            "vuln_func_body": vuln_body,
            "fixed_func_body": fixed_func['func_body'],
        })
    else:
        # Add the rejected function to the 'not_unit_testable' list for analysis
        not_unit_testable_builder.append({
            "func_name": vuln_func['func_name'],
            "fixed_func_idx": name,
            "function_prototype": "", # We don't have a proto, and don't need to parse again
            "cve_list": vuln_func['cve_list'],
            "cwe_list": vuln_func['cwe_list'],
            "vuln_func_body": vuln_body,
            "fixed_func_body": fixed_func['func_body'],
        })

unit_testable_df = pd.DataFrame(unit_testable_builder, columns=columns)
not_unit_testable_df = pd.DataFrame(not_unit_testable_builder, columns=columns) # Create the second DataFrame

print("Total pairs processed:", total_pairs_processed)
print("Unit testable samples found:", len(unit_testable_df))
print("NOT unit testable samples found:", len(not_unit_testable_df)) # Print the count for our debug set

# Save to a CSV
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)
unit_testable_df.to_csv(os.path.join(output_dir, "unit_testable_vuln_fixed_func_pairs.csv"), index=False)
not_unit_testable_df.to_csv(os.path.join(output_dir, "NOT_unit_testable_vuln_fixed_func_pairs.csv"), index=False) # Save the debug file

print(f"Saved curated datasets to {output_dir}")

100%|██████████| 10024/10024 [00:54<00:00, 185.42it/s]


Total pairs processed: 10024
Unit testable samples found: 525
NOT unit testable samples found: 9499
Saved curated datasets to data
